# DPO: learning from preference pairs

**Direct Preference Optimization: Your Language Model is Secretly a Reward Model** · Rafailov et al. · 2023

[Original paper](https://arxiv.org/abs/2305.18290)

**Learning goal:** Connect sequence log-probabilities to DPO and test masking, the reference, and update direction.

## Before starting

Causal LM, log_softmax, response masks, and introductory preferences.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

DPO converts chosen/rejected preferences into a loss without separately fitting a reward model in this objective. It uses a policy and fixed reference. Their probability ratio defines an implicit reward: chosen should beat rejected in that relative measure.

Increasing chosen probability alone is not the full diagnostic; the difference from reference matters. Preference datasets can be noisy and biased. We use three synthetic pairs and a tiny categorical policy so every quantity is visible.

## Reproduction boundary

DPO objective and masking only; no real language generation, preference corpus, or paper human-evaluation reproduction.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`margin = beta * [(logpi_ch - logpi_rej) - (logref_ch - logref_rej)]`; `loss = -logsigmoid(margin)`.

For an LM, logpi sums response-token log-probabilities after causal shifting and masking. Token means change the objective and length effects. Freeze the reference; beta scales the margin. Use log_softmax instead of log(softmax) for stability.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Check sequence log-probabilities. Modifying a masked token must not change the sum.

In [ ]:
def sequence_logp(logits, targets, mask):
    token_logp = logits.log_softmax(-1).gather(-1, targets[..., None]).squeeze(-1)
    return (token_logp * mask).sum(-1)
logits = torch.randn(2, 4, 7)
targets = torch.randint(0, 7, (2, 4)); mask = torch.tensor([[1, 1, 0, 0], [1, 1, 1, 0]])
changed = logits.clone(); changed[mask == 0] = torch.randn_like(changed[mask == 0]) * 10
assert torch.equal(sequence_logp(logits, targets, mask), sequence_logp(changed, targets, mask))
print('sequence log-probs:', sequence_logp(logits, targets, mask))

## Step 2

Initialize policy and reference identically: margin=0 and loss=log(2), even with unequal raw probabilities.

In [ ]:
policy = nn.Parameter(torch.tensor([[0.2, -0.3], [-0.2, 0.8], [0., 0.]]))
reference = policy.detach().clone().log_softmax(-1)
beta = 0.5
def dpo():
    logp = policy.log_softmax(-1)
    margins = beta * ((logp[:, 0] - logp[:, 1]) - (reference[:, 0] - reference[:, 1]))
    return -F.logsigmoid(margins).mean(), margins
loss, initial_margin = dpo()
assert torch.allclose(initial_margin, torch.zeros(3))
assert abs(loss.item() - math.log(2)) < 1e-6
loss.backward()
print('chosen/rejected gradients:', policy.grad)
assert (policy.grad[:, 0] < 0).all() and (policy.grad[:, 1] > 0).all()

## Step 3

Keep the reference fixed and observe growing margins. This plots the objective, not answer usefulness.

In [ ]:
frozen_reference = reference.clone()
optimizer = torch.optim.SGD([policy], lr=0.5)
history = []
for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss, margins = dpo(); loss.backward(); optimizer.step()
    history.append(loss.item())
assert torch.equal(reference, frozen_reference)
assert (dpo()[1] > initial_margin).all()
plt.plot(history, color='#d73535'); plt.ylabel('DPO loss'); plt.xlabel('step'); plt.show()
print('final relative margins:', dpo()[1].detach())

## Reading the result

Initial log(2) is a useful policy=reference regression check. Incorrect masks or an updated reference change training semantics even if loss falls. Real preference quality needs held-out prompt evaluation.

## Independent experiments

1. Recompute the reference after every step: what happens to the margin?
2. Swap chosen and rejected.
3. Compare sums and means for different response lengths.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

A reference constantly matched to the policy resets the relative margin. Swapping preference labels reverses the signal. Means alter length normalization; that is a different objective, not a universal DPO correction.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.